# Figure 1
TC count (a. NA, b. NWP, c. NEP) and TC lifetime (d. NA, e. NWP, f. NEP)

Import statements

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, weibull_min, pearsonr

## TC Counts
Plot the counts of TCs for basins: North Atlantic, West Pacific, East Pacific

### Filenames
Define lists of TempestExtremes TC track file names

In [2]:
start = 2004
stop = 2023
nens = 20

ts_or_td = "TD"

namelist_cf = [
    [
        f"/glade/work/smhenry/NeuralGCM/data/tracks/counterfactual/ens{X}_{YYYY}_JASO_TC_tracks_counterfactual.txt"
        for X in range(1, nens + 1)
    ]
    for YYYY in range(start, stop + 1)
]

namelist_f = [
    [
        f"/glade/work/smhenry/NeuralGCM/data/tracks/factual/ens{X}_{YYYY}_JASO_TC_tracks_factual.txt"
        for X in range(1, nens + 1)
    ]
    for YYYY in range(start, stop + 1)
]

namelist_ERA5 = [
    f"/glade/work/smhenry/NeuralGCM/data/tracks/ERA5/ERA5_{YYYY}_JASO_TC_tracks_vorticity.txt"
    for YYYY in range(start, stop + 1)
]

### Counting fuctions
Define functions for counting TC counts in NA, NWP, NEP

In [3]:
def count_NA(file):
    """
    counts the number of TCs from TempestExtremes output in the North Atlantic Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NA = False

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if is_in_NA:
                    count += 1
                is_in_NA = False
                in_TC = True
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if (
                        (285 <= lon <= 360 and 0 <= lat <= 50)
                        or (276 <= lon < 285 and 10 <= lat <= 50)
                        or (262 <= lon < 276 and 16.5 <= lat <= 50)
                    ):
                        is_in_NA = True

        if is_in_NA:
            count += 1

    return count


def count_NWP(file):
    """
    counts the number of TCs from TempestExtremes output in the Northwest Pacific Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NWP = False

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if is_in_NWP:
                    count += 1
                is_in_NWP = False
                in_TC = True
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if 100 <= lon <= 180 and 0 <= lat <= 50:  # need to update
                        is_in_NWP = True

        if is_in_NWP:
            count += 1

    return count


def count_NEP(file):
    """
    counts the number of TCs from TempestExtremes output in the Northeast Pacific Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NEP = False

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if is_in_NEP:
                    count += 1
                is_in_NEP = False
                in_TC = True
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if (
                        (275 <= lon <= 284 and 0 <= lat <= 10)
                        or (260 <= lon < 275 and 0 <= lat <= 16.5)
                        or (180 <= lon < 260 and 0 <= lat <= 50)
                    ):
                        is_in_NEP = True

        if is_in_NEP:
            count += 1

    return count


def count_TCs_ibtracs(year, basin, type):  # working yay!
    """
    returns the count of TCs from IBTrACS in a given basin for a given year
    input:
        year (int): desired year
        basin (str): desired basin
            keys: "AL", "WP", "EP"
        type (str): storm type
            keys: ts_or_td (tropical depression), "TS" (tropical storm), "HR" or "TY" (hurricane or typhoon, basin dependent)
            also see (DB: disturbance, TD: tropical depression, TS: tropical storm, TY: typhoon, ST: super typhoon, TC: tropical cyclone,
                    HU,HR: hurricane, SD: subtropical depression, SS: subtropical storm, EX: extratropical systems, PT: post tropical,
                    IN: inland, DS: dissipating, LO: low, WV: tropical wave, ET: extrapolated, MD: monsoon depression, XX: unknown)
    output:
        count (int): count of TCS
    criteria:
        TCs must last at least 54 hours to be consistent with TempestExtremes tracking
    """

    # open dataframe
    df = pd.read_csv(
        f"/glade/work/smhenry/NeuralGCM/data/ibtracs/IBTrACS_{year}_JASO.csv"
    )
    df["time"] = pd.to_datetime(df["time"])

    # Calculate storm durations
    storm_durations = df.groupby("stormid")["time"].agg(
        lambda x: (x.max() - x.min()).total_seconds() / 3600
    )

    # Keep only storms lasting at least 54 hours
    valid_storms = storm_durations[storm_durations >= 54].index
    df = df[df["stormid"].isin(valid_storms) & df["stormid"].str.contains(basin)]

    # Filter by storm type
    type_dict = {
        "TD": "HU|HR|TY|ST|TS|TC|SS|TD|SD",
        "TS": "HU|HR|TY|ST|TS|TC|SS",
        "HU": "HU|HR|TY|ST",
        "TY": "HU|HR|TY|ST",
    }

    if type in type_dict:
        df = df[df["type"].str.contains(type_dict[type])]

    return df["stormid"].nunique()

In [4]:
def count_NA_mo(file, mo):
    """
    counts the number of TCs from TempestExtremes output in the North Atlantic Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NA = False
    in_mo = False

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if (
                    is_in_NA and in_mo
                ):  # this addresses the last TC, then bools are reset
                    count += 1
                is_in_NA = False
                in_TC = True
                if line.split()[3] == mo:
                    in_mo = True
                else:
                    in_mo = False
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if (
                        (285 <= lon <= 360 and 0 <= lat <= 50)
                        or (276 <= lon < 285 and 10 <= lat <= 50)
                        or (262 <= lon < 276 and 16.5 <= lat <= 50)
                    ):
                        is_in_NA = True

        if is_in_NA and in_mo:
            count += 1

    return count


def count_NWP_mo(file, mo):
    """
    counts the number of TCs from TempestExtremes output in the Northwest Pacific Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NWP = False
    in_mo = None

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if (
                    is_in_NWP and in_mo
                ):  # this addresses the last TC, then bools are reset
                    count += 1
                is_in_NWP = False
                in_TC = True
                if line.split()[3] == mo:
                    in_mo = True
                else:
                    in_mo = False
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if 100 <= lon <= 180 and 0 <= lat <= 50:
                        is_in_NWP = True

        if is_in_NWP and in_mo:
            count += 1

    return count


def count_NEP_mo(file, mo):
    """
    counts the number of TCs from TempestExtremes output in the Northeast Pacific Basin
    input: file, a file name which is an output from TempestExtremes
    output: count, the number of TCs tracked in the file
    """
    count = 0
    in_TC = False
    is_in_NEP = False
    in_mo = None

    with open(file, "r") as df:
        for line in df:
            if line.startswith("start"):
                if (
                    is_in_NEP and in_mo
                ):  # this addresses the last TC, then bools are reset
                    count += 1
                is_in_NEP = False
                in_TC = True
                if line.split()[3] == mo:
                    in_mo = True
                else:
                    in_mo = False
            elif in_TC:
                data = line.split()
                if len(data) >= 4:
                    lon = float(data[2])
                    lat = float(data[3])

                    if (
                        (275 <= lon <= 284 and 0 <= lat <= 10)
                        or (260 <= lon < 275 and 0 <= lat <= 16.5)
                        or (180 <= lon < 260 and 0 <= lat <= 50)
                    ):
                        is_in_NEP = True

        if is_in_NEP and in_mo:
            count += 1

    return count


def count_TCs_ibtracs_mo(
    year, mo, basin, type
):  # updated to filter by storm start month
    """
    Returns the count of TCs from IBTrACS in a given basin for a specific year and month (based on storm start month).

    Inputs:
        year (int): desired year
        mo (int): desired month (1–12)
        basin (str): desired basin
            keys: "AL", "WP", "EP"
        type (str): storm type
            keys: "TD" (tropical depression), "TS" (tropical storm), "HR" or "TY" (hurricane or typhoon)
    Output:
        count (int): count of valid tropical cyclones (TCs)

    Criteria:
        - Storms must last at least 54 hours
        - Storm must start in the given month
    """
    # open dataframe
    df = pd.read_csv(
        f"/glade/work/smhenry/NeuralGCM/data/ibtracs/IBTrACS_{year}_JASO.csv"
    )

    df["time"] = pd.to_datetime(df["time"])

    # Calculate storm durations
    storm_durations = df.groupby("stormid")["time"].agg(
        lambda x: (x.max() - x.min()).total_seconds() / 3600
    )

    # Keep only storms lasting at least 54 hours
    valid_storms = storm_durations[storm_durations >= 54].index
    df_valid = df[df["stormid"].isin(valid_storms) & df["stormid"].str.contains(basin)]

    # Filter by storm type
    type_dict = {
        "TD": "HU|HR|TY|ST|TS|TC|SS|TD|SD",
        "TS": "HU|HR|TY|ST|TS|TC|SS",
        "HU": "HU|HR|TY|ST",
        "TY": "HU|HR|TY|ST",
    }

    if type in type_dict:
        df_type = df_valid[df_valid["type"].str.contains(type_dict[type])]

    # Get the first timestamp per storm
    storm_start_month = df_type.groupby("stormid")["time"].min().dt.month
    storms_starting_in_month = storm_start_month[storm_start_month == mo].index

    df_return = df_type[df_type["stormid"].isin(storms_starting_in_month)]

    return df_return["stormid"].nunique()

### Processing function
Define a function to process all files and return data to be plotted: ensmean_count_BASIN, twenty_BASIN, eighty_BASIN, min_BASIN, max_BASIN

In [5]:
def process_counts(namelist, basin, start, exclude=None, ens=False, min_max=False):
    """
    inputs:
        namelist: list of strs of filenames
        basin: str, "NA", "NWP", "NEP"
        start: start year of simulation data analysis
        exclude: list of sets of [[yr, ens]] to exclude
        ens: bool, contains ens data or not
        min_max: bool, return min max data for each year or not

    returns:
        ensmean_count: list of yearly ensemble mean TC counts, or if ens=False, just the yearly counts
        twenty: 20th percentile for each year, None if ens=False
        eighty: 80th percentile for each year, None if ens=False
        min: yearly min TC count, None if ens=False
        max: yearly max TC count, None if ens=False
    """

    nyr = np.shape(namelist)[0]
    if ens:
        nens = np.shape(namelist)[1]
    else:
        nens = 1

    # initalize array of TC counts for storage
    if ens:
        counts = np.zeros((nyr, nens))
    else:
        counts = np.zeros((nyr, 1))

    # count
    for iyr in range(nyr):
        if ens:
            for iens in range(nens):
                if exclude:
                    for i in range(len(exclude)):
                        if iyr == (exclude[i][0] - start) and iens == (
                            exclude[i][1] - 1
                        ):
                            counts[iyr][iens] = np.nan

                        else:
                            if basin == "NA":
                                counts[iyr][iens] = count_NA(namelist[iyr][iens])
                            elif basin == "NWP":
                                counts[iyr][iens] = count_NWP(namelist[iyr][iens])
                            elif basin == "NEP":
                                counts[iyr][iens] = count_NEP(namelist[iyr][iens])
                            else:
                                counts[iyr][iens] = np.nan

                else:
                    if basin == "NA":
                        counts[iyr][iens] = count_NA(namelist[iyr][iens])
                    elif basin == "NWP":
                        counts[iyr][iens] = count_NWP(namelist[iyr][iens])
                    elif basin == "NEP":
                        counts[iyr][iens] = count_NEP(namelist[iyr][iens])
                    else:
                        counts[iyr][iens] = np.nan

        else:
            if basin == "NA":
                counts[iyr] = count_NA(namelist[iyr])
            elif basin == "NWP":
                counts[iyr] = count_NWP(namelist[iyr])
            elif basin == "NEP":
                counts[iyr] = count_NEP(namelist[iyr])
            else:
                counts[iyr] = np.nan

    # take the ensemble mean and get a 20th and 80th percentile spread

    if ens:
        # initalize arrays for storage
        count = np.zeros((nyr, 1))
        twenty = np.zeros((nyr, 1))
        eighty = np.zeros((nyr, 1))
        min = np.zeros((nyr, 1))
        max = np.zeros((nyr, 1))

        for iyr in range(nyr):
            # count for that year
            yr_count = counts[iyr][:]

            # take mean across year
            count[iyr] = np.nanmean(yr_count)

            # take the 20th and 80th percentiles
            twenty[iyr] = np.nanpercentile(yr_count, 20)
            eighty[iyr] = np.nanpercentile(yr_count, 80)

            # take the mins and maxs
            if min_max:
                min[iyr] = np.nanmin(yr_count)
                max[iyr] = np.nanmax(yr_count)
            else:
                min = None
                max = None

    else:
        # initalize arrays for storage
        count = np.zeros((nyr, 1))
        twenty = np.zeros((nyr, 1))
        eighty = np.zeros((nyr, 1))
        min = np.zeros((nyr, 1))
        max = np.zeros((nyr, 1))

        for iyr in range(nyr):
            # count for that year
            yr_count = counts[iyr][:]

            # take mean across year
            count[iyr] = np.nanmean(yr_count)

            # take the 20th and 80th percentiles
            twenty[iyr] = np.nanpercentile(yr_count, 20)
            eighty[iyr] = np.nanpercentile(yr_count, 80)

            # take the mins and maxs
            if min_max:
                min[iyr] = np.nanmin(yr_count)
                max[iyr] = np.nanmax(yr_count)
            else:
                min = None
                max = None

    return count, twenty, eighty, min, max


def process_counts_mo(
    namelist, basin, start, mo, exclude=None, nens=1, min_max=False
):
    """
    inputs:
        namelist: list of strs of filenames
        basin: str, "NA", "NWP", "NEP"
        start: start year of simulation data analysis
        exclude: list of sets of [[yr, ens]] to exclude
        ens: bool, contains ens data or not
        min_max: bool, return min max data for each year or not

    returns:
        ensmean_count: list of yearly ensemble mean TC counts, or if ens=False, just the yearly counts
        twenty: 20th percentile for each year, None if ens=False
        eighty: 80th percentile for each year, None if ens=False
        min: yearly min TC count, None if ens=False
        max: yearly max TC count, None if ens=False
    """

    nyr = np.shape(namelist)[0]

    # initalize array of TC counts for storage
    if nens > 1:
        counts = np.zeros((nyr, nens))
    else:
        counts = np.zeros((nyr, 1))

    # count
    for iyr in range(nyr):
        if nens > 1:
            for iens in range(nens):
                if exclude:
                    for i in range(len(exclude)):
                        if iyr == (exclude[i][0] - start) and iens == (
                            exclude[i][1] - 1
                        ):
                            counts[iyr][iens] = np.nan

                        else:
                            if basin == "NA":
                                counts[iyr][iens] = count_NA_mo(namelist[iyr][iens], mo)
                            elif basin == "NWP":
                                counts[iyr][iens] = count_NWP_mo(
                                    namelist[iyr][iens], mo
                                )
                            elif basin == "NEP":
                                counts[iyr][iens] = count_NEP_mo(
                                    namelist[iyr][iens], mo
                                )
                            else:
                                counts[iyr][iens] = np.nan

                else:
                    if basin == "NA":
                        counts[iyr][iens] = count_NA_mo(namelist[iyr][iens], mo)
                    elif basin == "NWP":
                        counts[iyr][iens] = count_NWP_mo(namelist[iyr][iens], mo)
                    elif basin == "NEP":
                        counts[iyr][iens] = count_NEP_mo(namelist[iyr][iens], mo)
                    else:
                        counts[iyr][iens] = np.nan

        else:
            if basin == "NA":
                counts[iyr][iens] = count_NA_mo(namelist[iyr][iens], mo)
            elif basin == "NWP":
                counts[iyr][iens] = count_NWP_mo(namelist[iyr][iens], mo)
            elif basin == "NEP":
                counts[iyr][iens] = count_NEP_mo(namelist[iyr][iens], mo)
            else:
                counts[iyr][iens] = np.nan

    # take the ensemble mean and get a 20th and 80th percentile spread

    if nens > 1:
        # initalize arrays for storage
        count = np.zeros((nyr, 1))
        twenty = np.zeros((nyr, 1))
        eighty = np.zeros((nyr, 1))
        min = np.zeros((nyr, 1))
        max = np.zeros((nyr, 1))

        for iyr in range(nyr):
            # count for that year
            yr_count = counts[iyr][:]

            # take mean across year
            count[iyr] = np.nanmean(yr_count)

            # take the 20th and 80th percentiles
            twenty[iyr] = np.nanpercentile(yr_count, 20)
            eighty[iyr] = np.nanpercentile(yr_count, 80)

            # take the mins and maxs
            if min_max:
                min[iyr] = np.nanmin(yr_count)
                max[iyr] = np.nanmax(yr_count)
            else:
                min = None
                max = None

    else:
        # initalize arrays for storage
        count = np.zeros((nyr, 1))
        twenty = np.zeros((nyr, 1))
        eighty = np.zeros((nyr, 1))
        min = np.zeros((nyr, 1))
        max = np.zeros((nyr, 1))

        for iyr in range(nyr):
            # count for that year
            yr_count = counts[iyr][:]

            # take mean across year
            count[iyr] = np.nanmean(yr_count)

            # take the 20th and 80th percentiles
            twenty[iyr] = np.nanpercentile(yr_count, 20)
            eighty[iyr] = np.nanpercentile(yr_count, 80)

            # take the mins and maxs
            if min_max:
                min[iyr] = np.nanmin(yr_count)
                max[iyr] = np.nanmax(yr_count)
            else:
                min = None
                max = None

    return count, twenty, eighty, min, max

### Members to exclude

In [6]:
f_exclude = [[2005, 17], [2012, 14], [2017, 3], [2017, 12], [2017, 13], [2021, 16]]
cf_exclude = [[2009, 12], [2017, 15]]

### Processing

#### TC counts

In [7]:
# north atlantic
f_ensmean_count_NA, f_twenty_NA, f_eighty_NA, f_min_NA, f_max_NA = process_counts(
    namelist_f, "NA", start, ens=True, exclude=cf_exclude, min_max=True
)
cf_ensmean_count_NA, cf_twenty_NA, cf_eighty_NA, x, x = process_counts(
    namelist_cf, "NA", start, ens=True, exclude=f_exclude, min_max=False
)
ERA5_count_NA, x, x, x, x = process_counts(
    namelist_ERA5, "NA", start, ens=False, min_max=False
)

# northwest pacific
f_ensmean_count_NWP, f_twenty_NWP, f_eighty_NWP, f_min_NWP, f_max_NWP = process_counts(
    namelist_f, "NWP", start, ens=True, exclude=f_exclude, min_max=True
)
cf_ensmean_count_NWP, cf_twenty_NWP, cf_eighty_NWP, x, x = process_counts(
    namelist_cf, "NWP", start, ens=True, exclude=cf_exclude, min_max=False
)
ERA5_count_NWP, x, x, x, x = process_counts(
    namelist_ERA5, "NWP", start, ens=False, min_max=False
)

# northeast pacific
f_ensmean_count_NEP, f_twenHR_NEP, f_eighHR_NEP, f_min_NEP, f_max_NEP = process_counts(
    namelist_f, "NEP", start, ens=True, exclude=f_exclude, min_max=True
)
cf_ensmean_count_NEP, cf_twenHR_NEP, cf_eighHR_NEP, x, x = process_counts(
    namelist_cf, "NEP", start, ens=True, exclude=cf_exclude, min_max=False
)
ERA5_count_NEP, x, x, x, x = process_counts(
    namelist_ERA5, "NEP", start, ens=False, min_max=False
)

#### Observations

In [8]:
obs_TC_count_NA = []
obs_HR_count_NA = []
obs_TC_count_NEP = []
obs_TY_count_NEP = []
obs_TC_count_NWP = []
obs_TY_count_NWP = []

for year in range(start, stop + 1):
    obs_TC_count_NA.append(count_TCs_ibtracs(year, "AL", ts_or_td))
    obs_HR_count_NA.append(count_TCs_ibtracs(year, "AL", "HU"))
    obs_TC_count_NEP.append(count_TCs_ibtracs(year, "EP", ts_or_td))
    obs_TY_count_NEP.append(count_TCs_ibtracs(year, "EP", "TY"))
    obs_TC_count_NWP.append(count_TCs_ibtracs(year, "WP", ts_or_td))
    obs_TY_count_NWP.append(count_TCs_ibtracs(year, "WP", "TY"))

#### Correlations

In [9]:
# corr_TC_NA = np.corrcoef(f_ensmean_count_NA.flatten(), obs_TC_count_NA)[0, 1]
# corr_HR_NA = np.corrcoef(f_ensmean_count_NA.flatten(), obs_HR_count_NA)[0, 1]
# corr_ERA5_NA = np.corrcoef(f_ensmean_count_NA.flatten(), ERA5_count_NA.flatten())[0, 1]

# corr_TC_NEP = np.corrcoef(f_ensmean_count_NEP.flatten(), obs_TC_count_NEP)[0, 1]
# corr_HR_NEP = np.corrcoef(f_ensmean_count_NEP.flatten(), obs_TY_count_NEP)[0, 1]
# corr_ERA5_NEP = np.corrcoef(f_ensmean_count_NEP.flatten(), ERA5_count_NEP.flatten())[
#     0, 1
# ]

# corr_TC_NWP = np.corrcoef(f_ensmean_count_NWP.flatten(), obs_TC_count_NWP)[0, 1]
# corr_TY_NWP = np.corrcoef(f_ensmean_count_NWP.flatten(), obs_TY_count_NWP)[0, 1]
# corr_ERA5_NWP = np.corrcoef(f_ensmean_count_NWP.flatten(), ERA5_count_NWP.flatten())[
#     0, 1
# ]

In [10]:
corr_TC_NA, pval_TC_NA = pearsonr(f_ensmean_count_NA.flatten(), obs_TC_count_NA)
corr_HR_NA, pval_HR_NA = pearsonr(f_ensmean_count_NA.flatten(), obs_HR_count_NA)
corr_ERA5_NA, pval_ERA5_NA = pearsonr(f_ensmean_count_NA.flatten(), ERA5_count_NA.flatten())

corr_TC_NEP, pval_TC_NEP = pearsonr(f_ensmean_count_NEP.flatten(), obs_TC_count_NEP)
corr_HR_NEP, pval_HR_NEP = pearsonr(f_ensmean_count_NEP.flatten(), obs_TY_count_NEP)
corr_ERA5_NEP, pval_ERA5_NEP = pearsonr(f_ensmean_count_NEP.flatten(), ERA5_count_NEP.flatten())

corr_TC_NWP, pval_TC_NWP = pearsonr(f_ensmean_count_NWP.flatten(), obs_TC_count_NWP)
corr_TY_NWP, pval_TY_NWP = pearsonr(f_ensmean_count_NWP.flatten(), obs_TY_count_NWP)
corr_ERA5_NWP, pval_ERA5_NWP = pearsonr(f_ensmean_count_NWP.flatten(), ERA5_count_NWP.flatten())


print(f"{corr_TC_NA:.2f}, {pval_TC_NA:.2f}")
print(f"{corr_HR_NA:.2f}, {pval_HR_NA:.2f}")
print(f"{corr_ERA5_NA:.2f}, {pval_ERA5_NA:.2f}")

print(f"{corr_TC_NEP:.2f}, {pval_TC_NEP:.2f}")
print(f"{corr_HR_NEP:.2f}, {pval_HR_NEP:.2f}")
print(f"{corr_ERA5_NEP:.2f}, {pval_ERA5_NEP:.2f}")

print(f"{corr_TC_NWP:.2f}, {pval_TC_NWP:.2f}")
print(f"{corr_TY_NWP:.2f}, {pval_TY_NWP:.2f}")
print(f"{corr_ERA5_NWP:.2f}, {pval_ERA5_NWP:.2f}")

0.47, 0.04
0.42, 0.06
0.49, 0.03
0.61, 0.00
0.59, 0.01
0.65, 0.00
0.33, 0.15
0.42, 0.07
0.48, 0.03


## TC Lifetime Plots
Plot the PDF of TC lifetimes for factual and counterfactual simulations

Import statements

In [11]:
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gamma, weibull_min

Define simulations to analyze

In [12]:
start = 2004
stop = 2023
nens = 20

Namelists of tracks

In [13]:
# namelist of factual simulation tracks
namelist_f = [
    [
        f"/glade/work/smhenry/NeuralGCM/data/tracks/factual/ens{X}_{YYYY}_JASO_TC_tracks_factual.txt"
        for X in range(1, nens + 1)
    ]
    for YYYY in range(start, stop + 1)
]

namelist_cf = [
    [
        f"/glade/work/smhenry/NeuralGCM/data/tracks/counterfactual/ens{X}_{YYYY}_JASO_TC_tracks_counterfactual.txt"
        for X in range(1, nens + 1)
    ]
    for YYYY in range(start, stop + 1)
]

# namelist of ERA5 tracks
namelist_ERA5 = [
    [
        f"/glade/work/smhenry/NeuralGCM/data/tracks/ERA5/ERA5_{YYYY}_JASO_TC_tracks_vorticity.txt"
        for X in range(1, nens + 1)
    ]
    for YYYY in range(start, stop + 1)
]

f_exclude = [[2005, 17], [2012, 14], [2017, 3], [2017, 12], [2017, 13], [2021, 16]]
cf_exclude = [[2009, 12], [2017, 15]]

### Process data
Obtain PDFs of factual and counterfactual simulations

Lifetime function

In [14]:
def get_lifetime_NA(file):
    """
    returns a list of the TC lifetime of each track in file
    input: file, a file name which is an output from TempestExtremes
    output: lifetimes, a list of the lifetime [hours] of each track in file
        length of lifetimes should be equal to count
    """

    rows = []
    current_tc_id = None
    tc_tracks = {}

    with open(file, "r") as f:
        for line in f:
            if line.startswith("start"):
                current_tc_id = len(tc_tracks)
                tc_tracks[current_tc_id] = []
            else:
                data = line.split()
                if len(data) >= 10:
                    lon = float(data[2])
                    lat = float(data[3])
                    year, month, day, hour = map(int, data[6:10])
                    timestamp = datetime(year, month, day, hour)
                    tc_tracks[current_tc_id].append((lon, lat, timestamp))

    lifetimes = []
    for tc_id, track in tc_tracks.items():
        if not track:
            continue

        first_lon, first_lat, _ = track[0]
        if (
            ((first_lon >= 285) & (first_lon <= 360) & (first_lat >= 0))
            | ((first_lon >= 276) & (first_lon <= 285) & (first_lat >= 10))
            | ((first_lon >= 262) & (first_lon <= 276) & (first_lat >= 16.5))
        ):
            start_time = track[0][2]
            end_time = track[-1][2]
            lifetime_days = (end_time - start_time).total_seconds() / 3600 / 24
            lifetimes.append(int(lifetime_days))

    return lifetimes


def get_lifetime_NEP(file):
    """
    returns a list of the TC lifetime of each track in file
    input: file, a file name which is an output from TempestExtremes
    output: lifetimes, a list of the lifetime [hours] of each track in file
        length of lifetimes should be equal to count
    """

    rows = []
    current_tc_id = None
    tc_tracks = {}

    with open(file, "r") as f:
        for line in f:
            if line.startswith("start"):
                current_tc_id = len(tc_tracks)
                tc_tracks[current_tc_id] = []
            else:
                data = line.split()
                if len(data) >= 10:
                    lon = float(data[2])
                    lat = float(data[3])
                    year, month, day, hour = map(int, data[6:10])
                    timestamp = datetime(year, month, day, hour)
                    tc_tracks[current_tc_id].append((lon, lat, timestamp))

    lifetimes = []
    for tc_id, track in tc_tracks.items():
        if not track:
            continue

        first_lon, first_lat, _ = track[0]
        if (
            ((first_lon >= 275) & (first_lon <= 284) & (first_lat <= 10))
            | ((first_lon >= 260) & (first_lon <= 275) & (first_lat <= 16.5))
            | ((first_lon >= 180) & (first_lon <= 260) & (first_lat >= 0))
        ):
            start_time = track[0][2]
            end_time = track[-1][2]
            lifetime_days = (end_time - start_time).total_seconds() / 3600 / 24
            lifetimes.append(int(lifetime_days))

    return lifetimes


def get_lifetime_NWP(file):
    """
    returns a list of the TC lifetime of each track in file
    input: file, a file name which is an output from TempestExtremes
    output: lifetimes, a list of the lifetime [hours] of each track in file
        length of lifetimes should be equal to count
    """

    rows = []
    current_tc_id = None
    tc_tracks = {}

    with open(file, "r") as f:
        for line in f:
            if line.startswith("start"):
                current_tc_id = len(tc_tracks)
                tc_tracks[current_tc_id] = []
            else:
                data = line.split()
                if len(data) >= 10:
                    lon = float(data[2])
                    lat = float(data[3])
                    year, month, day, hour = map(int, data[6:10])
                    timestamp = datetime(year, month, day, hour)
                    tc_tracks[current_tc_id].append((lon, lat, timestamp))

    lifetimes = []
    for tc_id, track in tc_tracks.items():
        if not track:
            continue

        first_lon, first_lat, _ = track[0]
        if (first_lon >= 100) & (first_lon <= 180) & (first_lat >= 0):
            start_time = track[0][2]
            end_time = track[-1][2]
            lifetime_days = (end_time - start_time).total_seconds() / 3600 / 24
            lifetimes.append(int(lifetime_days))

    return lifetimes


def get_lifetime_IBTrACS(year, basin, type):
    # open dataframe
    df = pd.read_csv(
        f"/glade/work/smhenry/NeuralGCM/data/ibtracs/IBTrACS_{year}_JASO.csv"
    )

    df["time"] = pd.to_datetime(df["time"])

    # Calculate storm durations
    storm_durations = df.groupby("stormid")["time"].agg(
        lambda x: (x.max() - x.min()).total_seconds() / 3600
    )

    # Keep only storms lasting at least 54 hours
    valid_storms = storm_durations[storm_durations >= 54].index
    df_valid = df[df["stormid"].isin(valid_storms) & df["stormid"].str.contains(basin)]

    # Filter by storm type
    type_dict = {
        ts_or_td: "HU|HR|TY|ST|TS|TC|SS|TD|SD",
        "TS": "HU|HR|TY|ST|TS|TC|SS",
        "HU": "HU|HR|TY|ST",
        "TY": "HU|HR|TY|ST",
    }

    if type in type_dict:
        df_type = df_valid[df_valid["type"].str.contains(type_dict[type])]

    # Lifetimes
    lifetimes = df_type.groupby("stormid")["time"].agg(
        lambda x: (x.max() - x.min()).total_seconds() / 3600 / 24
    )

    lifetime_values = lifetimes.values

    return lifetime_values[np.where(lifetime_values > (54 / 24))]


values = np.arange(54 / 24, 24 + 54 / 24 + 1 / 24, 1 / 24)

Factual

In [15]:
# NA

# get lifetime data
f_lifetime_NA = []

for iyr, yr in enumerate(range(start, stop + 1)):
    for iens, ens in enumerate(range(1, nens + 1)):
        for i in range(len(f_exclude)):
            if iyr == (f_exclude[i][0] - start) and iens == (f_exclude[i][1] - 1):
                # exclude unfinished simulations
                f_lifetime_NA = f_lifetime_NA
            else:
                # update iyr, iens with lifetime
                f_lifetime_NA.extend(get_lifetime_NA(namelist_f[iyr][iens]))

# get the distribution
shape, loc, scale = weibull_min.fit(f_lifetime_NA)

# get the probabilities
prob_f_NA = weibull_min.pdf(values, shape, loc=loc, scale=scale)

# NEP

# get lifetime data
f_lifetime_NEP = []

for iyr, yr in enumerate(range(start, stop + 1)):
    for iens, ens in enumerate(range(1, nens + 1)):
        for i in range(len(f_exclude)):
            if iyr == (f_exclude[i][0] - start) and iens == (f_exclude[i][1] - 1):
                # exclude unfinished simulations
                f_lifetime_NEP = f_lifetime_NEP
            else:
                # update iyr, iens with lifetime
                f_lifetime_NEP.extend(get_lifetime_NEP(namelist_f[iyr][iens]))

# get the distribution
shape, loc, scale = weibull_min.fit(f_lifetime_NEP)

# get the probabilities
prob_f_NEP = weibull_min.pdf(values, shape, loc=loc, scale=scale)


# NWP

# get lifetime data
f_lifetime_NWP = []

for iyr, yr in enumerate(range(start, stop + 1)):
    for iens, ens in enumerate(range(1, nens + 1)):
        for i in range(len(f_exclude)):
            if iyr == (f_exclude[i][0] - start) and iens == (f_exclude[i][1] - 1):
                # exclude unfinished simulations
                f_lifetime_NWP = f_lifetime_NWP
            else:
                # update iyr, iens with lifetime
                f_lifetime_NWP.extend(get_lifetime_NWP(namelist_f[iyr][iens]))

# get the distribution
shape, loc, scale = weibull_min.fit(f_lifetime_NWP)

# get the probabilities
prob_f_NWP = weibull_min.pdf(values, shape, loc=loc, scale=scale)

Counterfactual

In [16]:
# NA

# get lifetime data
cf_lifetime_NA = []

for iyr, yr in enumerate(range(start, stop + 1)):
    for iens, ens in enumerate(range(1, nens + 1)):
        for i in range(len(cf_exclude)):
            if iyr == (cf_exclude[i][0] - start) and iens == (cf_exclude[i][1] - 1):
                # exclude unfinished simulations
                cf_lifetime_NA = cf_lifetime_NA
            else:
                # update iyr, iens with lifetime
                cf_lifetime_NA.extend(get_lifetime_NA(namelist_cf[iyr][iens]))

statistic_NA, p_value_NA = ks_2samp(cf_lifetime_NA, f_lifetime_NA, method="asymp")

# NEP

# get lifetime data
cf_lifetime_NEP = []

for iyr, yr in enumerate(range(start, stop + 1)):
    for iens, ens in enumerate(range(1, nens + 1)):
        for i in range(len(cf_exclude)):
            if iyr == (cf_exclude[i][0] - start) and iens == (cf_exclude[i][1] - 1):
                # exclude unfinished simulations
                cf_lifetime_NEP = cf_lifetime_NEP
            else:
                # update iyr, iens with lifetime
                cf_lifetime_NEP.extend(get_lifetime_NEP(namelist_cf[iyr][iens]))

statistic_NEP, p_value_NEP = ks_2samp(cf_lifetime_NEP, f_lifetime_NEP)

# NWP

# get lifetime data
cf_lifetime_NWP = []

for iyr, yr in enumerate(range(start, stop + 1)):
    for iens, ens in enumerate(range(1, nens + 1)):
        for i in range(len(cf_exclude)):
            if iyr == (cf_exclude[i][0] - start) and iens == (cf_exclude[i][1] - 1):
                # exclude unfinished simulations
                cf_lifetime_NWP = cf_lifetime_NWP
            else:
                # update iyr, iens with lifetime
                cf_lifetime_NWP.extend(get_lifetime_NWP(namelist_cf[iyr][iens]))

statistic_NWP, p_value_NWP = ks_2samp(cf_lifetime_NWP, f_lifetime_NWP)

### IBTrACS

In [17]:
obs_lifetime_NA = []
obs_lifetime_NEP = []
obs_lifetime_NWP = []

for year in range(start, stop + 1):
    obs_lifetime_NA.extend(get_lifetime_IBTrACS(year, "AL", ts_or_td))
    obs_lifetime_NEP.extend(get_lifetime_IBTrACS(year, "EP", ts_or_td))
    obs_lifetime_NWP.extend(get_lifetime_IBTrACS(year, "WP", ts_or_td))

obs_lifetime_NA = np.array(obs_lifetime_NA)
obs_lifetime_NEP = np.array(obs_lifetime_NEP)
obs_lifetime_NWP = np.array(obs_lifetime_NWP)

# get the distribution
shape, loc, scale = weibull_min.fit(obs_lifetime_NA)
# get the probabilities
prob_obs_NA = weibull_min.pdf(values, shape, loc=loc, scale=scale)

# get the distribution
shape, loc, scale = weibull_min.fit(obs_lifetime_NEP)
# get the probabilities
prob_obs_NEP = weibull_min.pdf(values, shape, loc=loc, scale=scale)

# get the distribution
shape, loc, scale = weibull_min.fit(obs_lifetime_NWP)
# get the probabilities
prob_obs_NWP = weibull_min.pdf(values, shape, loc=loc, scale=scale)

## Seasonality

In [ ]:
mo_list = ["7", "8", "9", "10"]

mo_count_NA_f = []
mo_count_NWP_f = []
mo_count_NEP_f = []

for mo in mo_list:
    # north atlantic
    this_mo_count_NA, twenty_NA, eighty_NA, min_NA, max_NA = process_counts_mo(
        namelist_f,
        "NA",
        start,
        mo,
        nens=20
,
        min_max=False,
        exclude=f_exclude,
    )
    this_mo_count_NA = this_mo_count_NA / f_ensmean_count_NA
    twenty_NA = twenty_NA / f_ensmean_count_NA
    eighty_NA = eighty_NA / f_ensmean_count_NA
    if min_NA is not None and max_NA is not None:
        min_NA = min_NA / f_ensmean_count_NA
        max_NA = max_NA / f_ensmean_count_NA
    else:
        min_NA, max_NA = None, None
        
    # northwest pacific
    this_mo_count_NWP, twenty_NWP, eighty_NWP, min_NWP, max_NWP = process_counts_mo(
        namelist_f,
        "NWP",
        start,
        mo,
        nens=20
,
        min_max=False,
        exclude=f_exclude,
    )
    this_mo_count_NWP = this_mo_count_NWP / f_ensmean_count_NWP
    twenty_NWP = twenty_NWP / f_ensmean_count_NWP
    eighty_NWP = eighty_NWP / f_ensmean_count_NWP
    if min_NWP is not None and max_NA is not None:
        min_NWP = min_NWP / f_ensmean_count_NWP
        max_NWP = max_NWP / f_ensmean_count_NWP
    else:
        min_NWP, max_NWP = None, None
        
    # northeast pacific
    this_mo_count_NEP, twenty_NEP, eighty_NEP, min_NEP, max_NEP = process_counts_mo(
        namelist_f,
        "NEP",
        start,
        mo,
        nens=20
,
        min_max=False,
        exclude=f_exclude,
    )
    this_mo_count_NEP = this_mo_count_NEP / f_ensmean_count_NEP
    twenty_NEP = twenty_NEP / f_ensmean_count_NEP
    eighty_NEP = eighty_NEP / f_ensmean_count_NEP
    if min_NEP is not None and max_NA is not None:
        min_NEP = min_NEP / f_ensmean_count_NEP
        max_NEP = max_NEP / f_ensmean_count_NEP
    else:
        min_NEP, max_NEP = None, None

    mo_count_NA_f.append(this_mo_count_NA.flatten())
    mo_count_NWP_f.append(this_mo_count_NWP.flatten())
    mo_count_NEP_f.append(this_mo_count_NEP.flatten())

In [ ]:
mo_list = ["7", "8", "9", "10"]

mo_count_NA_cf = []
mo_count_NWP_cf = []
mo_count_NEP_cf = []

for mo in mo_list:
    # north atlantic
    this_mo_count_NA, twenty_NA, eighty_NA, min_NA, max_NA = process_counts_mo(
        namelist_cf,
        "NA",
        start,
        mo,
        nens=20,
        min_max=False,
        exclude=cf_exclude,
    )
    this_mo_count_NA = this_mo_count_NA / cf_ensmean_count_NA
    twenty_NA = twenty_NA / cf_ensmean_count_NA
    eighty_NA = eighty_NA / cf_ensmean_count_NA
    if min_NA is not None and max_NA is not None:
        min_NA = min_NA / cf_ensmean_count_NA
        max_NA = max_NA / cf_ensmean_count_NA
    else:
        min_NA, max_NA = None, None
        
    # northwest pacific
    this_mo_count_NWP, twenty_NWP, eighty_NWP, min_NWP, max_NWP = process_counts_mo(
        namelist_cf,
        "NWP",
        start,
        mo,
        nens=20,
        min_max=False,
        exclude=cf_exclude,
    )
    this_mo_count_NWP = this_mo_count_NWP / cf_ensmean_count_NWP
    twenty_NWP = twenty_NWP / cf_ensmean_count_NWP
    eighty_NWP = eighty_NWP / cf_ensmean_count_NWP
    if min_NWP is not None and max_NA is not None:
        min_NWP = min_NWP / cf_ensmean_count_NWP
        max_NWP = max_NWP / cf_ensmean_count_NWP
    else:
        min_NWP, max_NWP = None, None

    # northeast pacific
    this_mo_count_NEP, twenty_NEP, eighty_NEP, min_NEP, max_NEP = process_counts_mo(
        namelist_cf,
        "NEP",
        start,
        mo,
        nens=20,
        min_max=False,
        exclude=cf_exclude,
    )
    this_mo_count_NEP = this_mo_count_NEP / cf_ensmean_count_NEP
    twenty_NEP = twenty_NEP / cf_ensmean_count_NEP
    eighty_NEP = eighty_NEP / cf_ensmean_count_NEP
    if min_NEP is not None and max_NA is not None:
        min_NEP = min_NEP / cf_ensmean_count_NEP
        max_NEP = max_NEP / cf_ensmean_count_NEP
    else:
        min_NEP, max_NEP = None, None
        
    mo_count_NA_cf.append(this_mo_count_NA.flatten())
    mo_count_NWP_cf.append(this_mo_count_NWP.flatten())
    mo_count_NEP_cf.append(this_mo_count_NEP.flatten())

In [ ]:
obs_mo_count_NA = [[], [], [], []]
obs_mo_count_NEP = [[], [], [], []]
obs_mo_count_NWP = [[], [], [], []]

mo_list = [7, 8, 9, 10]

for year in range(start, stop + 1):
    for i, mo in enumerate(mo_list):
        obs_mo_count_NA[i].append(count_TCs_ibtracs_mo(year, mo, "AL", ts_or_td))
        obs_mo_count_NEP[i].append(count_TCs_ibtracs_mo(year, mo, "EP", ts_or_td))
        obs_mo_count_NWP[i].append(count_TCs_ibtracs_mo(year, mo, "WP", ts_or_td))

obs_mo_count_NA = obs_mo_count_NA / np.array(obs_TC_count_NA).T
obs_mo_count_NEP = obs_mo_count_NEP / np.array(obs_TC_count_NEP).T
obs_mo_count_NWP = obs_mo_count_NWP / np.array(obs_TC_count_NWP).T

## Plot

In [ ]:
yrlist = np.arange(start, stop + 1)

fig, axs = plt.subplots(ncols=3, nrows=3, figsize=(8, 8))
axs = axs.flatten()

"""
TC COUNTS
"""

"""
NA
"""

ax = axs[0]

# factual
ax.plot(
    yrlist,
    f_ensmean_count_NA,
    color="tab:orange",
    label="Factual",
    linewidth=1,
    marker="s",
    markersize=3,
)
ax.fill_between(
    yrlist,
    f_twenty_NA.flatten(),
    f_eighty_NA.flatten(),
    color="tab:orange",
    alpha=0.25,
)

for iyr in range(stop - start + 1):
    ax.vlines(
        yrlist[iyr],
        f_min_NA[iyr],
        f_max_NA[iyr],
        color="tab:orange",
        capstyle="round",
        linewidth=0.75,
    )
    ax.hlines(
        f_min_NA[iyr],
        yrlist[iyr] - 0.1,
        yrlist[iyr] + 0.1,
        color="tab:orange",
        linewidth=0.75,
    )
    ax.hlines(
        f_max_NA[iyr],
        yrlist[iyr] - 0.1,
        yrlist[iyr] + 0.1,
        color="tab:orange",
        linewidth=0.75,
    )

# counterfactual
ax.plot(
    yrlist,
    cf_ensmean_count_NA,
    color="tab:green",
    label="Counterfactual",
    linewidth=1,
    marker="o",
    markersize=3,
)
ax.fill_between(
    yrlist,
    cf_twenty_NA.flatten(),
    cf_eighty_NA.flatten(),
    color="tab:green",
    alpha=0.25,
)

# ERA5
ax.plot(
    yrlist,
    ERA5_count_NA,
    color="tab:blue",
    marker="P",
    markersize=3,
    label="ERA5",
    linewidth=1,
)

# observed HRs
ax.plot(
    yrlist,
    obs_HR_count_NA,
    color="tab:grey",
    markersize=3,
    linewidth=1,
    marker="^",
    label="Obs. HRs/TYs",
)

# observed TCs
ax.plot(
    yrlist,
    obs_TC_count_NA,
    color="k",
    marker="v",
    markersize=3,
    linewidth=1,
    label="Obs. TCs",
)

# correlation coefficient
ax.text(
    -2.295,
    3,
    f"Corr (Factual, Obs. TCs): {corr_TC_NA:.2f}*\nCorr (Factual, Obs. HRs): {corr_HR_NA:.2f}\nCorr (Factual, ERA5): {corr_ERA5_NA:.2f}*",
    transform=plt.gca().transAxes,
    fontsize=8,
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="grey"),
)

ax.set_ylim([0, 26])
ax.set_xlabel("Year", fontsize=10)
ax.set_ylabel("Count", fontsize=10)
ax.set_xticks([2004, 2008, 2012, 2016, 2020, 2024])
ax.set_title("(a) NA TC Count", fontsize=14)
ax.tick_params(axis="both", which="major", labelsize=10)
ax.tick_params(axis="both", which="minor", labelsize=10)
ax.legend(
    loc="upper left",
    bbox_to_anchor=(3.5, 0.95),
    fontsize=10,
    frameon=True,
)
ax.grid(alpha=0.3)

"""
NWP
"""

ax = axs[2]

# factual
ax.plot(
    yrlist,
    f_ensmean_count_NWP,
    color="tab:orange",
    label="Factual",
    linewidth=1,
    marker="s",
    markersize=3,
)
ax.fill_between(
    yrlist,
    f_twenty_NWP.flatten(),
    f_eighty_NWP.flatten(),
    color="tab:orange",
    alpha=0.25,
)

for iyr in range(stop - start + 1):
    ax.vlines(
        yrlist[iyr],
        f_min_NWP[iyr],
        f_max_NWP[iyr],
        color="tab:orange",
        capstyle="round",
        linewidth=0.75,
    )
    ax.hlines(
        f_min_NWP[iyr],
        yrlist[iyr] - 0.1,
        yrlist[iyr] + 0.1,
        color="tab:orange",
        linewidth=0.75,
    )
    ax.hlines(
        f_max_NWP[iyr],
        yrlist[iyr] - 0.1,
        yrlist[iyr] + 0.1,
        color="tab:orange",
        linewidth=0.75,
    )

# counterfactual
ax.plot(
    yrlist,
    cf_ensmean_count_NWP,
    color="tab:green",
    label="Counterfactual",
    linewidth=1,
    marker="o",
    markersize=3,
)
ax.fill_between(
    yrlist,
    cf_twenty_NWP.flatten(),
    cf_eighty_NWP.flatten(),
    color="tab:green",
    alpha=0.25,
)

# ERA5
ax.plot(
    yrlist,
    ERA5_count_NWP,
    color="tab:blue",
    marker="P",
    markersize=3,
    label="ERA5",
    linewidth=1,
)

# observed HRs
ax.plot(
    yrlist,
    obs_TY_count_NWP,
    color="tab:grey",
    markersize=3,
    linewidth=1,
    marker="^",
    label="Obs. HRs/TYs",
)

# observed TCs
ax.plot(
    yrlist,
    obs_TC_count_NWP,
    color="k",
    marker="v",
    markersize=3,
    linewidth=1,
    label="Obs. TCs",
)

# correlation coefficient
ax.text(
    0.105,
    3,
    f"Corr (Factual, Obs. TCs): {corr_TC_NWP:.2f}\nCorr (Factual, Obs. TYs): {corr_TY_NWP:.2f}\nCorr (Factual, ERA5): {corr_ERA5_NWP:.2f}*",
    transform=plt.gca().transAxes,
    fontsize=8,
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="grey"),
)

ax.set_ylim([0, 26])
ax.set_xlabel("Year", fontsize=10)
ax.set_xticks([2004, 2008, 2012, 2016, 2020, 2024])
ax.tick_params(axis="both", which="major", labelsize=10)
ax.tick_params(axis="both", which="minor", labelsize=10)
ax.set_title("(c) NWP TC Count", fontsize=14)

ax.grid(alpha=0.3)

"""
NEP
"""

ax = axs[1]

# factual
ax.plot(
    yrlist,
    f_ensmean_count_NEP,
    color="tab:orange",
    label="Factual",
    linewidth=1,
    marker="s",
    markersize=3,
)
ax.fill_between(
    yrlist,
    f_twenHR_NEP.flatten(),
    f_eighHR_NEP.flatten(),
    color="tab:orange",
    alpha=0.25,
)

for iyr in range(stop - start + 1):
    ax.vlines(
        yrlist[iyr],
        f_min_NEP[iyr],
        f_max_NEP[iyr],
        color="tab:orange",
        capstyle="round",
        linewidth=0.75,
    )
    ax.hlines(
        f_min_NEP[iyr],
        yrlist[iyr] - 0.1,
        yrlist[iyr] + 0.1,
        color="tab:orange",
        linewidth=0.75,
    )
    ax.hlines(
        f_max_NEP[iyr],
        yrlist[iyr] - 0.1,
        yrlist[iyr] + 0.1,
        color="tab:orange",
        linewidth=0.75,
    )

# counterfactual
ax.plot(
    yrlist,
    cf_ensmean_count_NEP,
    color="tab:green",
    label="Counterfactual",
    linewidth=1,
    marker="o",
    markersize=3,
)
ax.fill_between(
    yrlist,
    cf_twenHR_NEP.flatten(),
    cf_eighHR_NEP.flatten(),
    color="tab:green",
    alpha=0.25,
)

# ERA5
ax.plot(
    yrlist,
    ERA5_count_NEP,
    color="tab:blue",
    marker="P",
    markersize=3,
    label="ERA5",
    linewidth=1,
)

# observed HRs
ax.plot(
    yrlist,
    obs_TY_count_NEP,
    color="tab:grey",
    markersize=3,
    linewidth=1,
    marker="^",
    label="Obs. HRs/TYs",
)

# observed TCs
ax.plot(
    yrlist,
    obs_TC_count_NEP,
    color="k",
    marker="v",
    markersize=3,
    linewidth=1,
    label="Obs. TCs",
)

# correlation coefficient
ax.text(
    -1.095,
    3,
    f"Corr (Factual, Obs. TCs): {corr_TC_NEP:.2f}*\nCorr (Factual, Obs. TYs): {corr_HR_NEP:.2f}*\nCorr (Factual, ERA5): {corr_ERA5_NEP:.2f}*",
    transform=plt.gca().transAxes,
    fontsize=8,
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="grey"),
)

ax.set_ylim([0, 26])
ax.set_xlabel("Year", fontsize=10)
ax.set_xticks([2004, 2008, 2012, 2016, 2020, 2024])
ax.tick_params(axis="both", which="major", labelsize=10)
ax.tick_params(axis="both", which="minor", labelsize=10)
ax.set_title("(b) NEP TC Count", fontsize=14)
ax.grid(alpha=0.3)

"""
TC LIFETIMES
"""

"""
NA
"""

ax = axs[3]

# plot factual
# ax.plot(values, prob_f_NA, "tab:orange", label="factual", linewidth=2)
ax.hist(
    f_lifetime_NA, bins=np.arange(0, 22, 1), density=True, color="tab:orange", alpha=0.5
)
ax.hist(
    f_lifetime_NA,
    bins=np.arange(0, 22, 1),
    density=True,
    color="tab:orange",
    histtype="step",
    linewidth=1.5,
)

# plot counterfactual
# ax.plot(values, prob_cf_NA, "tab:green", label="counterfactual", linewidth=2)
ax.hist(
    cf_lifetime_NA, bins=np.arange(0, 22, 1), density=True, color="tab:green", alpha=0.5
)
ax.hist(
    cf_lifetime_NA,
    bins=np.arange(0, 22, 1),
    density=True,
    color="tab:green",
    histtype="step",
    linewidth=1.5,
)

# plot obs
# ax.plot(values, prob_obs_NA, "k", label="TCs", linewidth=2)
ax.hist(
    obs_lifetime_NA,
    bins=np.arange(0, 22, 1),
    density=True,
    histtype="step",
    color="k",
    linewidth=1.5,
)
# ax.text(
#     -1.85,
#     2.6,
#     f"p(F, CF)={p_value_NA:.2f}",
#     transform=plt.gca().transAxes,
#     fontsize=10,
#     bbox=dict(facecolor="white", alpha=0.7, edgecolor="grey"),
# )

# arrange figure
ax.grid(alpha=0.3)
ax.set_xlim(2.25, values[-1])
ax.set_ylim(0, 0.25)
ax.set_xlabel("Lifetime (days)", fontsize=10)
ax.set_ylabel("Density (normalized)", fontsize=10)
ax.set_title("(d) NA Lifetimes", fontsize=14)
ax.tick_params(axis="both", which="major", labelsize=10)
ax.tick_params(axis="both", which="minor", labelsize=10)

"""
NWP
"""

ax = axs[5]

# plot factual
# ax.plot(values, prob_f_NWP, "tab:orange", label="factual", linewidth=2)
ax.hist(
    f_lifetime_NWP,
    bins=np.arange(0, 22, 1),
    density=True,
    color="tab:orange",
    alpha=0.5,
)
ax.hist(
    f_lifetime_NWP,
    bins=np.arange(0, 22, 1),
    density=True,
    color="tab:orange",
    histtype="step",
    linewidth=1.5,
)

# plot counterfactual
# ax.plot(values, prob_cf_NWP, "tab:green", label="counterfactual", linewidth=2)
ax.hist(
    cf_lifetime_NWP,
    bins=np.arange(0, 22, 1),
    density=True,
    color="tab:green",
    alpha=0.5,
)
ax.hist(
    cf_lifetime_NWP,
    bins=np.arange(0, 22, 1),
    density=True,
    color="tab:green",
    histtype="step",
    linewidth=1.5,
)

# plot obs
# ax.plot(values, prob_obs_NWP, "k", label="TCs", linewidth=2)
ax.hist(
    obs_lifetime_NWP,
    bins=np.arange(0, 22, 1),
    density=True,
    histtype="step",
    color="k",
    linewidth=1.5,
)
# ax.text(
#     0.55,
#     2.6,
#     f"p(F, CF)={p_value_NWP:.2f}",
#     transform=plt.gca().transAxes,
#     fontsize=10,
#     bbox=dict(facecolor="white", alpha=0.7, edgecolor="grey"),
# )

# arrange figure
ax.grid(alpha=0.3)
ax.set_xlim(2.25, values[-1])
ax.set_ylim(0, 0.25)
ax.set_xlabel("Lifetime (days)", fontsize=10)
ax.set_title("(f) NWP Lifetimes", fontsize=14)
ax.tick_params(axis="both", which="major", labelsize=10)
ax.tick_params(axis="both", which="minor", labelsize=10)


"""
NEP
"""
ax = axs[4]

# plot factual
# ax.plot(values, prob_f_NEP, "tab:orange", label="factual", linewidth=2)
ax.hist(
    f_lifetime_NEP,
    bins=np.arange(0, 22, 1),
    density=True,
    color="tab:orange",
    alpha=0.5,
)
ax.hist(
    f_lifetime_NEP,
    bins=np.arange(0, 22, 1),
    density=True,
    color="tab:orange",
    histtype="step",
    linewidth=1.5,
)

# plot counterfactual
# ax.plot(values, prob_cf_NEP, "tab:green", label="counterfactual", linewidth=2)
ax.hist(
    cf_lifetime_NEP,
    bins=np.arange(0, 22, 1),
    density=True,
    color="tab:green",
    alpha=0.5,
)
ax.hist(
    cf_lifetime_NEP,
    bins=np.arange(0, 22, 1),
    density=True,
    color="tab:green",
    histtype="step",
    linewidth=1.5,
)

# plot obs
# ax.plot(values, prob_obs_NEP, "k", label="TCs", linewidth=2)
ax.hist(
    obs_lifetime_NEP,
    bins=np.arange(0, 22, 1),
    density=True,
    histtype="step",
    color="k",
    linewidth=1.5,
)

# ax.text(
#     -0.65,
#     2.6,
#     f"p(F, CF)={p_value_NEP:.2f}",
#     transform=plt.gca().transAxes,
#     fontsize=10,
#     bbox=dict(facecolor="white", alpha=0.7, edgecolor="grey"),
# )

# arrange figure
ax.grid(alpha=0.3)
ax.set_xlim(2.25, values[-1])
ax.set_ylim(0, 0.25)
ax.set_xlabel("Lifetime (days)", fontsize=10)
ax.set_title("(e) NEP Lifetimes", fontsize=14)
ax.tick_params(axis="both", which="major", labelsize=10)
ax.tick_params(axis="both", which="minor", labelsize=10)


"""
Seasonality
"""

months = ["Jul", "Aug", "Sept", "Oct"]

positions = []
group_spacing = 3  # space between groups
intra_group_spacing = 0.6  # space between boxes within a group

for i in range(len(months)):
    start_spacing = i * group_spacing
    positions.extend(
        [
            start_spacing,
            start_spacing + intra_group_spacing,
            start_spacing + 2 * intra_group_spacing,
        ]
    )

# Use midpoint between first and last box in each group for x-ticks
mid_positions = [
    (positions[i] + positions[i + 2]) / 2 for i in range(0, len(positions), 3)
]


ax = axs[6]

data = []
for i in range(len(months)):
    data.extend(
        [
            mo_count_NA_cf[i],
            mo_count_NA_f[i],
            obs_mo_count_NA[i],
        ]
    )


boxes = ax.boxplot(
    data,
    positions=positions,
    patch_artist=True,
    medianprops=dict(color="tab:blue", linewidth=2),
    showfliers=False,
)

for i, box in enumerate(boxes["boxes"]):
    box.set_linewidth(1.25)
    if i % 3 == 0:
        box.set_facecolor("tab:green")
    elif i % 3 == 1:
        box.set_facecolor("tab:orange")
    elif i % 3 == 2:
        box.set_facecolor("none")

ax.set_xticks(mid_positions)
ax.set_xticklabels(months, fontsize=10)
ax.set_yticks([0, .2, .4, .6,])
ax.set_yticklabels([0, .2, .4, .6,], fontsize=10)
ax.set_ylim(0, .75)
ax.set_ylabel("Count", fontsize=10)
ax.set_title("(g) NA Monthly Count", fontsize=14)
ax.grid(alpha=0.3)


"""
NWP
"""
ax = axs[8]

data = []
for i in range(len(months)):
    data.extend(
        [
            mo_count_NWP_cf[i],
            mo_count_NWP_f[i],
            obs_mo_count_NWP[i],
        ]
    )  # now three per month


boxes = ax.boxplot(
    data,
    positions=positions,
    patch_artist=True,
    medianprops=dict(color="tab:blue", linewidth=2),
    showfliers=False,
)

for i, box in enumerate(boxes["boxes"]):
    box.set_linewidth(1.25)
    if i % 3 == 0:
        box.set_facecolor("tab:green")
    elif i % 3 == 1:
        box.set_facecolor("tab:orange")
    elif i % 3 == 2:
        box.set_facecolor("none")

ax.set_xticks(mid_positions)
ax.set_xticklabels(months, fontsize=10)
ax.set_yticks([0, .2, .4, .6,])
ax.set_yticklabels([0, .2, .4, .6,], fontsize=10)
ax.set_ylim(0, .75)
ax.set_title("(i) NWP Monthly Count", fontsize=14)
ax.grid(alpha=0.3)

"""
NEP
"""
ax = axs[7]

data = []
for i in range(len(months)):
    data.extend(
        [
            mo_count_NEP_cf[i],
            mo_count_NEP_f[i],
            obs_mo_count_NEP[i],
        ]
    )  # now three per month

boxes = ax.boxplot(
    data,
    positions=positions,
    patch_artist=True,
    medianprops=dict(color="tab:blue", linewidth=2),
    showfliers=False,
)

for i, box in enumerate(boxes["boxes"]):
    box.set_linewidth(1.25)
    if i % 3 == 0:
        box.set_facecolor("tab:green")
    elif i % 3 == 1:
        box.set_facecolor("tab:orange")
    elif i % 3 == 2:
        box.set_facecolor("none")


ax.set_xticks(mid_positions)
ax.set_xticklabels(months, fontsize=10)
ax.set_yticks([0, .2, .4, .6,])
ax.set_yticklabels([0, .2, .4, .6,], fontsize=10)
ax.set_ylim(0, .75)
ax.set_title("(h) NEP Monthly Count", fontsize=14)
ax.grid(alpha=0.3)

plt.subplots_adjust(left=0, bottom=0, right=1, top=0.875, wspace=0.2, hspace=0.76)
plt.savefig("./figs/figure1_w_seasonality.png", dpi=600, bbox_inches="tight")
plt.savefig("./figs/figure1_w_seasonality.pdf", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
np.sum(np.array(mo_count_NA_f), axis=0)

In [ ]:
np.sum(np.array(mo_count_NA_cf), axis=0)

In [ ]:
np.sum(np.array(obs_mo_count_NA), axis=0)